# In Class Activity April 14th, 2026

In [1]:
pip install optuna

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 28.7 MB/s  0:00:00

   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ------ --------------------------------- 1/6 [greenlet]
   ------ --------------------------------- 1/6 [greenlet]
   ------ --------------------------------- 1/6 [greenlet]
   ------ --------------------------------- 1/6 [greenlet]
   ------------- -------------------------- 2/6 [colorlog]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 

### Importing libraries, preparing data, initial EDA

In [2]:
# importing libraries (feel free to add more if you want to explore other things)
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import optuna


In [3]:
# importing data
adult = pd.read_csv("C:\\Users\\2003n\\Downloads\\adult.csv")
adult.head(20)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K
6,29,?,227026,HS-grad,9,Never-married,?,Unmarried,Black,Male,0,0,40,United-States,<=50K
7,63,Self-emp-not-inc,104626,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,>50K
8,24,Private,369667,Some-college,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,<=50K
9,55,Private,104996,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,<=50K


In [4]:
# initial EDA with sweetviz
report = sv.analyze(adult)
report.show_html('sweet_report.html')

# you are welcome to replace this cell with your own EDA, but make sure to include
# some visualizations and insights about the data


                                             |          | [  0%]   00:00 -> (? left)

Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### In the markdown cell below describe what you learned from your EDA and how it will inform your modeling decisions





The Data seems to be skewed, theres an imbalance, there are duplicate data points, and there is missing data points. 

### Data Preprocessing (minimal) and Baseline Model

In [5]:
# data cleaning and preprocessing

# changing ? to NaN
adult = adult.replace('?', np.nan)

#education and education num are redundant, so we can drop one of them
adult = adult.drop('education', axis=1)

# target variable is income with 2 levels, so we can encode it as 0 and 1
adult['income'] = adult['income'].apply(lambda x: 1 if x == '>50K' else 0)

# change dtype categorical variables to category
categorical_cols = adult.select_dtypes(include='object').columns
adult[categorical_cols] = adult[categorical_cols].astype('category')


adult.head(20)

,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0
5,34,Private,198693,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,0
6,29,NaN,227026,9,Never-married,NaN,Unmarried,Black,Male,0,0,40,United-States,0
7,63,Self-emp-not-inc,104626,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,1
8,24,Private,369667,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,0
9,55,Private,104996,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,0


In [6]:
# defining features and target variable
X = adult.drop('income', axis=1)
y = adult['income']

# splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, 
                                                    random_state=42, stratify=y)

In [7]:
# building xgboost default model and evaluating with stratified k-fold cross validation
xgb_cv = XGBClassifier(enable_categorical=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1')
print(f'Cross-validated F1 scores: {cv_scores}')
print(f'Mean F1 score: {cv_scores.mean()}') 


Cross-validated F1 scores: [0.70680507 0.70892566 0.70898981 0.72424942 0.71086556]
Mean F1 score: 0.7119671046056588


### Use the markdown cell below to describe your baseline model performance and how you will try to improve it

The performance of the model is fine, since the F1-score is 0.71, it means that it balances precision and recall well. I will try to improve it by hypertuning its parameters

### Model feature exploration

In the code cell below, explore different features of XGBoost and how they work (e.g. scale_pos_weight, max_depth, learning_rate).
Use stratified k-fold cross or repeated stratified k-fold cross validation with your model building. 
You should explore at least 3 different features of XGBoost.
Identify the model that performs best.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


models = {
    "model_1": XGBClassifier(
        enable_categorical=True,
        random_state=42,
        max_depth=3,
        learning_rate=0.1,
        scale_pos_weight=1,
        n_estimators=100
    ),
    "model_2": XGBClassifier(
        enable_categorical=True,
        random_state=42,
        max_depth=5,
        learning_rate=0.05,
        scale_pos_weight=1,
        n_estimators=200
    ),
    "model_3": XGBClassifier(
        enable_categorical=True,
        random_state=42,
        max_depth=4,
        learning_rate=0.2,
        scale_pos_weight=2,
        n_estimators=150
    )  
}

results = []

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=skf, scoring="f1")
    results.append({
        "model": name,
        "mean_f1": scores.mean(),
        "std_f1": scores.std()
    })
    print(f"{name} F1 scores: {scores}")
    print(f"{name} mean F1: {scores.mean():.4f}\n")

results_df = pd.DataFrame(results).sort_values(by="mean_f1", ascending=False)
print("Best model from feature exploration:")
print(results_df)


model_1 F1 scores: [0.68200335 0.69602681 0.68998563 0.69165659 0.68460606]
model_1 mean F1: 0.6889

model_2 F1 scores: [0.70252418 0.71782763 0.70527307 0.71533192 0.7092803 ]
model_2 mean F1: 0.7100

model_3 F1 scores: [0.72301946 0.72616712 0.73035926 0.7387495  0.72455322]
model_3 mean F1: 0.7286

Best model from feature exploration:
     model   mean_f1    std_f1
2  model_3  0.728570  0.005649
1  model_2  0.710047  0.005804
0  model_1  0.688856  0.005011


In [ ]:
The best model

### Tuning with GridSearchCV

Use the code cell below to set up your parameter grid and run GridSearchCV with your preferred model from above. You should tune 4-5 hyperparameters utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [9]:
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import classification_report, f1_score

# split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42, stratify=y
)

# parameter grid
param_grid = {
    "max_depth": [3, 4, 5],
    "learning_rate": [0.05, 0.1, 0.2],
    "n_estimators": [100, 150, 200],
    "scale_pos_weight": [1, 2, 3],
    "subsample": [0.8, 1.0]
}

# base model
xgb_model = XGBClassifier(
    enable_categorical=True,
    random_state=42
)

# grid search
grid = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring="f1",
    cv=skf,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best cross-validated F1:", grid.best_score_)

# final model predictions
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("\nTest F1 score:", f1_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Fitting 5 folds for each of 162 candidates, totalling 810 fits
Best parameters: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'scale_pos_weight': 2, 'subsample': 1.0}
Best cross-validated F1: 0.7286942478730701

Test F1 score: 0.7292035398230089

Classification Report:

              precision    recall  f1-score   support

           0       0.93      0.88      0.90      7431
           1       0.67      0.79      0.73      2338

    accuracy                           0.86      9769
   macro avg       0.80      0.84      0.82      9769
weighted avg       0.87      0.86      0.86      9769



The best XGBoost setting was a learning rate = 0.1, max_depth = 5, n_estimators = 200, scale_post_weight = 2, and supsample =1.0. This is shown with a CV F1 score of 0.787 and a test F1 score of 0.729. 

### Tuning with RandomizedSearchCV

Using the code cell below as a starting point, tune your preferred model from above. Tune the same 4-5 hyperparameters from above utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [11]:
# split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42, stratify=y
)

# parameter distributions
param_dist = {
    "max_depth": [3, 4, 5, 6],
    "learning_rate": np.linspace(0.01, 0.2, 10),
    "n_estimators": [100, 150, 200, 250],
    "scale_pos_weight": [1, 2, 3, 4],
    "subsample": [0.8, 0.9, 1.0]
}

# base model
xgb_model = XGBClassifier(
    enable_categorical=True,
    random_state=42
)

# randomized search
random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=20,
    scoring="f1",
    cv=skf,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

random_search.fit(X_train, y_train)

print("Best parameters:", random_search.best_params_)
print("Best cross-validated F1:", random_search.best_score_)

# final model predictions
best_random_model = random_search.best_estimator_
y_pred_random = best_random_model.predict(X_test)

print("\nTest F1 score:", f1_score(y_test, y_pred_random))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_random))


Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters: {'subsample': 0.9, 'scale_pos_weight': 2, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': np.float64(0.11555555555555555)}
Best cross-validated F1: 0.7239261110453395

Test F1 score: 0.7251968503937007

Classification Report:

              precision    recall  f1-score   support

           0       0.93      0.88      0.90      7431
           1       0.67      0.79      0.73      2338

    accuracy                           0.86      9769
   macro avg       0.80      0.83      0.81      9769
weighted avg       0.87      0.86      0.86      9769



The best XGBoost setting from RandomizedSearchCV was subsample = 0.9, scale_pos_weight = 2, n_estimators = 200, max_depth = 3, and learning_rate = 0.116. This is shown by a CV F1 score of 0.724 and a test F1 score of 0.725.

### Tuning with Optuna

Using the code cell below as a starting point, tune your preferred model from above. You should tune the same 4-5 parameters as above using cross validation. Train a final model using the best hyperparameters and report your model performance.

In [12]:

def objective(trial):
    scale_pos_weight = trial.suggest_float('scale_pos_weight', 1.0, 10.0)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    n_estimators = trial.suggest_int('n_estimators', 100, 250, step=50)
    subsample = trial.suggest_float('subsample', 0.8, 1.0)

    xgb_optuna = XGBClassifier(
        random_state=42,
        scale_pos_weight=scale_pos_weight,
        max_depth=max_depth,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        subsample=subsample,
        enable_categorical=True
    )

    cv_scores = cross_val_score(xgb_optuna, X, y, cv=skf, scoring='f1')
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f'Best parameters from Optuna: {study.best_params}')
print(f'Best F1 score from Optuna: {study.best_value}')

xgb_optuna_best = XGBClassifier(
    random_state=42,
    scale_pos_weight=study.best_params['scale_pos_weight'],
    max_depth=study.best_params['max_depth'],
    learning_rate=study.best_params['learning_rate'],
    n_estimators=study.best_params['n_estimators'],
    subsample=study.best_params['subsample'],
    enable_categorical=True
)

xgb_optuna_best.fit(X_train, y_train)
y_pred_optuna = xgb_optuna_best.predict(X_test)

print("\nTest F1 score:", f1_score(y_test, y_pred_optuna))
print(f'Classification report for Optuna-tuned model:\n{classification_report(y_test, y_pred_optuna)}')

[I 2026-04-15 23:48:08,221] A new study created in memory with name: no-name-0c4f4581-bb2d-4b6e-9a54-13aef1fb0428


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-04-15 23:48:11,251] Trial 0 finished with value: 0.6527102734088917 and parameters: {'scale_pos_weight': 9.254370452256838, 'max_depth': 7, 'learning_rate': 0.026010079821318755, 'n_estimators': 200, 'subsample': 0.944978883249664}. Best is trial 0 with value: 0.6527102734088917.
[I 2026-04-15 23:48:12,610] Trial 1 finished with value: 0.7004739289841535 and parameters: {'scale_pos_weight': 4.590984222220145, 'max_depth': 6, 'learning_rate': 0.2788267265058906, 'n_estimators': 100, 'subsample': 0.8769118159784227}. Best is trial 1 with value: 0.7004739289841535.
[I 2026-04-15 23:48:15,061] Trial 2 finished with value: 0.6858958872946898 and parameters: {'scale_pos_weight': 7.209959689951171, 'max_depth': 8, 'learning_rate': 0.09465289729514982, 'n_estimators': 150, 'subsample': 0.9356370618651115}. Best is trial 1 with value: 0.7004739289841535.
[I 2026-04-15 23:48:16,806] Trial 3 finished with value: 0.6999222374086622 and parameters: {'scale_pos_weight': 1.0252901956124991, '

The best XGBoost setting from Optuna was scale_pos_weight = 1.85, max_depth = 5, learning_rate = 0.274, n_estimators = 150, and subsample = 0.957. This is shown by a CV F1 score of 0.726 and a test F1 score of 0.730.

### Tuning results

In the markdown cell below describe your experience tuning with the different methods. Which produced the best results? Which do you prefer?


Out of the three tuning methods, GridSearchCV produced the best cross-validation results with a CV F1 score of 0.787 and a test F1 score of 0.729. RandomizedSearchCV gave a CV F1 score of 0.724 and a test F1 score of 0.725, while Optuna gave a CV F1 score of 0.726 and a test F1 score of 0.730. Even though GridSearchCV had the strongest cross-validation score, I liked Optuna the most because it was more flexible and efficient while still giving the highest test F1 score